In [1]:
import os
os.chdir('..')

In [2]:
from scipy.io import loadmat
import numpy as np
from scipy.signal import resample_poly
from src.models.onnx_inference import ONNXModel
import time
from src.features.cwt import cwt2scalogram
from src.features.stft import stft2spectrogram
from scipy.special import softmax
import matplotlib.pyplot as plt

In [3]:
def _load_signal(path: str) -> np.ndarray:
    mat = loadmat(path)
    for k in mat:
        if k.startswith("__"): continue
        if "DE_time" in k:
            return np.asarray(mat[k]).squeeze()
    raise ValueError(f"No DE_time key found in {path}. Keys: {[k for k in mat if not k.startswith('__')]}")

In [18]:
signal_path = r"D:\Capstone\dataset\raw\CWRU\12k_Drive_End_Bearing_Fault_Data\B\021\225_3.mat"
model_path = r"D:\Capstone\models\ONNX\CNN\SCALOGRAM\MobileNetV4\best_model_int8.onnx"
WIN, HOP = 2048, 2048
FS = 12000

sig    = _load_signal(signal_path)
normal = "Normal" in signal_path

if normal:
    sig = resample_poly(sig, up=1, down=4)
    
n = len(sig)
total_windows = max(0, (n - WIN) // HOP + 1)

model = ONNXModel(model_path)

In [22]:
for start in range(0, n - WIN + 1, HOP):
    end = start + WIN
    window = sig[start:end].copy()

    img = cwt2scalogram(window, img_size=128, gray=True)

    img = np.expand_dims(img.astype(np.float32) / 255.0, axis=0)

    t0 = time.perf_counter()
    logits = model.predict(img)
    inf_ms = (time.perf_counter() - t0) * 1000

    prob = softmax(logits, axis=1).squeeze()
    label = int(np.argmax(prob))
    confidence = float(np.max(prob))

    print(label, confidence, f"{inf_ms:.2f} ms")

3 0.9669570922851562 0.83 ms
3 0.9458422660827637 0.63 ms
3 0.9331268668174744 0.64 ms
3 0.9334485530853271 0.54 ms
3 0.9724340438842773 0.52 ms
3 0.9850091934204102 0.52 ms
3 0.9809509515762329 0.64 ms
3 0.9683065414428711 0.62 ms
3 0.9284353852272034 0.57 ms
3 0.936220645904541 0.70 ms
3 0.9061048626899719 0.70 ms
3 0.915579617023468 0.46 ms
3 0.9004382491111755 0.53 ms
3 0.9768149256706238 0.60 ms
3 0.9696418046951294 0.73 ms
3 0.876993715763092 0.62 ms
3 0.9118192195892334 0.62 ms
3 0.9720219969749451 0.52 ms
3 0.9672971367835999 0.68 ms
3 0.9844431281089783 0.64 ms
3 0.8826903700828552 0.54 ms
3 0.9526404738426208 0.61 ms
3 0.9382533431053162 0.58 ms
3 0.9662367105484009 0.58 ms
3 0.9521294236183167 0.61 ms
3 0.9716757535934448 0.59 ms
3 0.8939195275306702 0.52 ms
3 0.9638303518295288 0.55 ms
3 0.9759977459907532 0.55 ms
3 0.9168995022773743 0.62 ms
3 0.95685213804245 0.57 ms
3 0.9172932505607605 0.61 ms
3 0.9749626517295837 0.52 ms
3 0.8745968341827393 0.53 ms
3 0.904247283935546